In [197]:
# math librairies
import numpy as np
from scipy import ndimage

from plotly import express as px
import pandas               # used with plotly
import plotly.graph_objects as go

# 2D image processing 
import cv2

# custom librairies
import imicpe
import imicpe.optim as optim
print(imicpe.__version__)

1.0.14


# **Initialisation**

## Fonction de coût

In [198]:
def E(x, lam):
    return np.sum(x**2) + lam * np.sum(np.abs(x))

def grad_f(x):
    return 2 * x

def subgradient_l1(x):
    g = np.sign(x)
    g[x == 0] = 0.0
    return g


In [199]:
def prox_l1(x, gamma, lam):
    return np.sign(x) * np.maximum(np.abs(x) - gamma*lam, 0.0)


def prox_l2(x, gamma):
    return x / (1 + 2*gamma)

def prox_f(x, gamma, lam):
    """
    Proximal de f(x) = x^2 + lam |x|
    """
    y = prox_l2(x, gamma)
    return prox_l1(y, gamma, lam)


def moreau_l1(x, gamma):
    """
    Enveloppe de Moreau de |x|
    """
    x = np.asarray(x)
    M = np.zeros_like(x)

    mask = np.abs(x) <= gamma
    M[mask]  = (x[mask]**2) / (2*gamma)
    M[~mask] = np.abs(x[~mask]) - gamma/2

    return M


# **Algorithme**

In [200]:
# paramètres
Niter = 10
gamma = 1.21955        # doit vérifier gamma < 1/L avec L=2
lam   = 1.0
N     = 50

# initialisation
xn = np.zeros((Niter+1, N))
En = np.zeros(Niter+1)

xn[0] = np.random.randn(N)
En[0] = E(xn[0], lam)

# itérations forward-backward
for it in range(Niter):

    # étape forward : descente de gradient sur f
    y = xn[it] - gamma * grad_f(xn[it])

    # étape backward : prox sur g
    xn[it+1] = prox_l1(y, gamma, lam)

    # énergie
    En[it+1] = E(xn[it+1], lam)

xhat = xn[-1]


In [201]:
df = pandas.DataFrame({
    "itération": np.arange(Niter+1),
    "Énergie": En
})

fig = px.line(
    df,
    x="itération",
    y="Énergie",
    title="Convergence – Algorithme de gradient proximal (N-D)",
    width=700,
    height=400
)
fig.show()


## **Affichage des résultats**

In [202]:
x = np.linspace(-3, 3, 400)
gamma = 3
z = 1.0

lam_plot = 1.0

df = pandas.DataFrame({
    "x": x,
    "prox_|x|": prox_l1(x, gamma, lam_plot)
})

fig = px.line(
    df,
    x="x",
    y= "prox_|x|",
    title=f"Opérateur proximal (γ = {gamma})",
    labels={"value": "prox", "x": "x"},
    width=800,
    height=400
)
fig.show()


plt_cost = pandas.DataFrame({
    'x': np.arange(1, len(En)+1),
    'y': En,
})

fig = px.line(
    plt_cost,
    x='x',
    y='y',
    log_x=True,
    log_y=True,
    labels={'x':'itérations (log)','y':'E(x^k) (log)'},
    title='Évolution de la fonction de coût',
    width=800,
    height=350
)
fig.show()

df = pandas.DataFrame({
    "x": x,
    "|x|": np.abs(x),
    "M_gamma|x|": moreau_l1(x, gamma)
})

fig = px.line(
    df,
    x="x",
    y=["|x|", "M_gamma|x|"],
    title=f"Enveloppe de Moreau de |x| (γ = {gamma})",
    width=800,
    height=400
)
fig.show()



In [203]:
# résultat    
if N == 1:
    xn = xn.flatten()
    data = pandas.DataFrame({'x':xn, 'y':np.abs(xn), 'iter':np.arange(Niter+1)})
    
    fig = px.scatter(data, x='x', y='y',  
                color='iter',       
                labels={'x':'xn[it]','y':'|xn[it]|'},
                # size='iter',
                )
    
    t = np.linspace(-(np.abs(xn[0])+.5),np.abs(xn[0])+.5,20)
    fig.add_traces(px.line(x=t, y=np.abs(t), color_discrete_sequence=['black'] ).data)

    fig.update_layout(coloraxis_colorbar=dict(
        title=dict(text="Numéro d'itération")))
    t = np.linspace(-4, 4, 400)
    f_t = t**2 + lam*np.abs(t)

    fig = px.line(x=t, y=f_t, labels={'x':'x','y':'f(x)'})
    fig.add_scatter(x=xn, y=xn**2 + lam*np.abs(xn),
                    mode="markers+lines",
                    name="Itérations")


else: 
    data = pandas.DataFrame({'x':np.arange(N), 'y':xn[0], 'iteration':'0'})
    for it in range(0,Niter,Niter//10):
        plt_xn = pandas.DataFrame({'x':np.arange(N), 'y':xn[it], 'iteration':str(it)})
        data   = pandas.concat([data,plt_xn]) 

    myscale = px.colors.sample_colorscale(
        colorscale=px.colors.sequential.Plasma,
        samplepoints=len(data['iteration'].unique()),
        low=0.0, high=1.0,
        colortype="rgb",
    )

    fig = px.line(data,
                x='x',
                y='y', 
                color='iteration',    #color_discrete_sequence=['black','orange', 'orangered'],
                color_discrete_sequence=myscale,           
                labels={'x':'indices','y':''},
                # title='Signaux', 
                markers=True,
                width=800, height=450)


fig.show()